In [19]:
# ============================================================
# TASK 1: DATA EXPLORATION AND ENRICHMENT
# Ethiopia Financial Inclusion Forecast Project
# ============================================================


# ============================
# 1. Import Libraries
# ============================

import pandas as pd
import numpy as np
import os
from datetime import datetime



# ============================
# 2. Load Raw Data
# ============================

RAW_PATH = "../data/raw/"
PROCESSED_PATH = "../data/processed/"

os.makedirs(PROCESSED_PATH, exist_ok=True)


data = pd.read_csv(
    RAW_PATH + "ethiopia_fi_unified_data.csv"
)


reference_codes = pd.read_csv(
    RAW_PATH + "reference_codes.csv"
)


guide = pd.read_csv(
    RAW_PATH + "Additional Data Points Guide.csv"
)



print("Data loaded")
print("Main data:", data.shape)
print("Reference codes:", reference_codes.shape)
print("Guide:", guide.shape)



# ============================
# 3. Dataset Understanding
# ============================

print("\nColumns:")
print(data.columns.tolist())


print("\nDataset Preview:")
display(data.head())


print("\nDataset Information:")
data.info()



print("\nMissing Values:")
display(
    data.isnull()
    .sum()
    .sort_values(ascending=False)
)



# ============================
# 4. Record Analysis
# ============================


for col in [
    "record_type",
    "pillar",
    "source_type",
    "confidence"
]:

    if col in data.columns:

        print("\n", col)

        display(
            data[col]
            .value_counts(dropna=False)
        )



# ============================
# 5. Date Analysis
# ============================


date_columns = [
    "observation_date",
    "event_date",
    "target_date"
]


for col in date_columns:

    if col in data.columns:

        data[col] = pd.to_datetime(
            data[col],
            errors="coerce"
        )

        print("\n", col)

        print(
            "Start:",
            data[col].min()
        )

        print(
            "End:",
            data[col].max()
        )



# ============================
# 6. Indicator Analysis
# ============================


if "indicator_code" in data.columns:

    print("\nIndicators")

    display(
        data["indicator_code"]
        .unique()
    )


    print(
        "Number of indicators:",
        data["indicator_code"].nunique()
    )


    print("\nIndicator coverage")

    display(
        data.groupby(
            "indicator_code"
        )
        .size()
        .sort_values(
            ascending=False
        )
    )



# ============================
# 7. Observation Analysis
# ============================


observations = data[
    data["record_type"]=="observation"
]


print(
    "Observations:",
    len(observations)
)


display(
    observations.head()
)



# ============================
# 8. Event Analysis
# ============================


events = data[
    data["record_type"]=="event"
]


print(
    "Events:",
    len(events)
)


display(
    events.head()
)



if "category" in events.columns:

    print("\nEvent categories")

    display(
        events["category"]
        .value_counts(dropna=False)
    )



# ============================
# 9. Impact Links Review
# ============================


impact_file = RAW_PATH + "impact_links.csv"


if os.path.exists(impact_file):

    impact_links = pd.read_csv(
        impact_file
    )

    print(
        "Impact links:",
        impact_links.shape
    )

    display(
        impact_links.head()
    )

else:

    impact_links = pd.DataFrame()

    print(
        "No impact_links.csv found - creating new links"
    )



# ============================================================
# 10. DATA ENRICHMENT
# ============================================================


collection_date = datetime.today().strftime("%Y-%m-%d")

collector = "Your Name"



# ----------------------------
# New Observations
# ----------------------------


new_observations = pd.DataFrame([

{
"record_type":"observation",
"pillar":"access",
"indicator":"Mobile money account ownership",
"indicator_code":"MOBILE_MONEY_ACCESS",
"value_numeric":46,
"observation_date":"2023-01-01",
"source_name":"World Bank Global Findex",
"source_url":"https://www.worldbank.org/",
"confidence":"high",
"original_text":"Percentage of adults with mobile money accounts",
"collected_by":collector,
"collection_date":collection_date,
"notes":"Useful for forecasting digital financial inclusion adoption"
}

])



# ----------------------------
# New Events
# ----------------------------


new_events = pd.DataFrame([

{
"record_type":"event",
"category":"policy",
"event_date":"2021-01-01",
"description":"Expansion of Ethiopia digital financial services initiatives",
"source_name":"National Bank of Ethiopia",
"source_url":"https://www.nbe.gov.et/",
"confidence":"medium",
"original_text":"Digital financial services expansion policy initiatives",
"collected_by":collector,
"collection_date":collection_date,
"notes":"Policy changes may influence financial inclusion growth"
}

])



# ----------------------------
# New Impact Links
# ----------------------------


new_impact_links = pd.DataFrame([

{
"parent_id":"policy_2021_digital_finance",
"pillar":"access",
"related_indicator":"MOBILE_MONEY_ACCESS",
"impact_direction":"positive",
"impact_magnitude":"high",
"lag_months":12,
"evidence_basis":"Digital finance policies increase access channels"
}

])



# ============================
# 11. Merge Data
# ============================


enriched_data = pd.concat(
    [
        data,
        new_observations,
        new_events
    ],
    ignore_index=True
)



# ============================
# 12. Save Outputs
# ============================


enriched_data.to_csv(
    PROCESSED_PATH+
    "ethiopia_fi_enriched_data.csv",
    index=False
)



new_observations.to_csv(
    PROCESSED_PATH+
    "new_observations.csv",
    index=False
)


new_events.to_csv(
    PROCESSED_PATH+
    "new_events.csv",
    index=False
)


new_impact_links.to_csv(
    PROCESSED_PATH+
    "new_impact_links.csv",
    index=False
)



reference_codes.to_csv(
    PROCESSED_PATH+
    "reference_codes_copy.csv",
    index=False
)



# ============================
# 13. Create Enrichment Log
# ============================


log = f"""
# Data Enrichment Log

## Collection Date
{collection_date}


## Added Observation

Indicator:
Mobile money account ownership

Source:
World Bank Global Findex

Confidence:
High

Reason:
Added digital financial inclusion measurement useful for forecasting.


## Added Event

Event:
Digital financial services expansion initiatives

Source:
National Bank of Ethiopia

Confidence:
Medium

Reason:
Policy events influence financial inclusion trends.


## Added Impact Link

Indicator:
MOBILE_MONEY_ACCESS

Direction:
Positive

Lag:
12 months

Evidence:
Digital finance policies improve access and usage.
"""


with open(
    "../data_enrichment_log.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(log)



print("""
====================================
TASK 1 COMPLETED SUCCESSFULLY

Created:
✓ enriched dataset
✓ new observations
✓ new events
✓ impact links
✓ enrichment log

Remaining:
Git branch + commit + Pull Request
====================================
""")

Data loaded
Main data: (43, 34)
Reference codes: (71, 4)
Guide: (18, 8)

Columns:
['record_id', 'record_type', 'category', 'pillar', 'indicator', 'indicator_code', 'indicator_direction', 'value_numeric', 'value_text', 'value_type', 'unit', 'observation_date', 'period_start', 'period_end', 'fiscal_year', 'gender', 'location', 'region', 'source_name', 'source_type', 'source_url', 'confidence', 'related_indicator', 'relationship_type', 'impact_direction', 'impact_magnitude', 'impact_estimate', 'lag_months', 'evidence_basis', 'comparable_country', 'collected_by', 'collection_date', 'original_text', 'notes']

Dataset Preview:


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,%,2014-12-31,NaN,NaN,2014,all,national,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,%,2017-12-31,NaN,NaN,2017,all,national,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,%,2021-12-31,NaN,NaN,2021,all,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,%,2021-12-31,NaN,NaN,2021,male,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,%,2021-12-31,NaN,NaN,2021,female,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN



Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   record_id            43 non-null     str    
 1   record_type          43 non-null     str    
 2   category             10 non-null     str    
 3   pillar               33 non-null     str    
 4   indicator            43 non-null     str    
 5   indicator_code       43 non-null     str    
 6   indicator_direction  33 non-null     str    
 7   value_numeric        33 non-null     float64
 8   value_text           10 non-null     str    
 9   value_type           43 non-null     str    
 10  unit                 33 non-null     str    
 11  observation_date     43 non-null     str    
 12  period_start         10 non-null     str    
 13  period_end           10 non-null     str    
 14  fiscal_year          43 non-null     str    
 15  gender               43 non-nul

lag_months             43
region                 43
impact_estimate        43
relationship_type      43
evidence_basis         43
notes                  43
related_indicator      43
impact_magnitude       43
impact_direction       43
category               33
period_start           33
collection_date        33
period_end             33
value_text             33
source_url             12
indicator_direction    10
pillar                 10
value_numeric          10
original_text          10
unit                   10
record_type             0
record_id               0
indicator_code          0
indicator               0
value_type              0
observation_date        0
location                0
gender                  0
fiscal_year             0
confidence              0
source_type             0
source_name             0
comparable_country      0
collected_by            0
dtype: int64


 record_type


...

Events: 10


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
33,EVT_0001,event,product_launch,NaN,Telebirr Launch,EVT_TELEBIRR,NaN,NaN,Launched,categorical,NaN,2021-05-17,NaN,NaN,2021,all,national,NaN,Ethio Telecom,operator,https://www.ethiotelecom.et/,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,First major mobile money service in Ethiopia,NaN
34,EVT_0002,event,market_entry,NaN,Safaricom Ethiopia Commercial Launch,EVT_SAFARICOM,NaN,NaN,Launched,categorical,NaN,2022-08-01,NaN,NaN,2022,all,national,NaN,News,news,NaN,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,End of state telecom monopoly,NaN
35,EVT_0003,event,product_launch,NaN,M-Pesa Ethiopia Launch,EVT_MPESA,NaN,NaN,Launched,categorical,NaN,2023-08-01,NaN,NaN,2023,all,national,NaN,Safaricom,operator,NaN,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Second mobile money entrant,NaN
36,EVT_0004,event,infrastructure,NaN,Fayda Digital ID Program Rollout,EVT_FAYDA,NaN,NaN,Launched,categorical,NaN,2024-01-01,NaN,NaN,2024,all,national,NaN,NIDP,regulator,https://www.id.gov.et/,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,National biometric digital ID system,NaN
37,EVT_0005,event,policy,NaN,Foreign Exchange Liberalization,EVT_FX_REFORM,NaN,NaN,Implemented,categorical,NaN,2024-07-29,NaN,NaN,2024,all,national,NaN,NBE,regulator,NaN,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Birr float introduced,NaN



Event categories


category
product_launch    2
infrastructure    2
policy            2
market_entry      1
milestone         1
partnership       1
pricing           1
Name: count, dtype: int64

No impact_links.csv found - creating new links

TASK 1 COMPLETED SUCCESSFULLY

Created:
✓ enriched dataset
✓ new observations
✓ new events
✓ impact links
✓ enrichment log

Remaining:
Git branch + commit + Pull Request

